<a href="https://colab.research.google.com/github/lsteffenel/M2Atmo_et_Climat/blob/main/Tutoriel_OpenSTL_FR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/lsteffenel/OpenSTL/blob/OpenSTL-Lightning/examples/tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutoriel OpenSTL — Prédire l'évolution d'images en séquence

**À quoi sert ce notebook ?**

OpenSTL est une bibliothèque qui apprend à un réseau de neurones à **prédire des images futures** à partir d'images passées. Concrètement : on montre au modèle 6 images (par exemple, 6 trames d'une vidéo satellite), et il doit deviner les 6 images suivantes. C'est le même principe que la prévision numérique du temps — remplacer les équations physiques par de l'apprentissage — appliqué ici à de la vidéo.

Ce tutoriel déroule les 4 étapes d'un projet complet, du début à la fin :

1. **Préparer les données** : convertir des vidéos en tableaux numériques exploitables ;
2. **Organiser les données** pour l'entraînement (train / validation / test) ;
3. **Entraîner et évaluer** un modèle fourni par OpenSTL ;
4. **Visualiser** les prédictions du modèle face aux images réelles.

Aucune connaissance avancée en programmation n'est requise : il suffit d'exécuter les cellules dans l'ordre, et de modifier uniquement les quelques valeurs signalées par un commentaire `# À ADAPTER`.

## Étape 0 — Installation de l'environnement

La première cellule télécharge le code source d'OpenSTL depuis GitHub et l'installe. La seconde installe les bibliothèques dont il dépend. C'est long (quelques minutes) mais il ne faut le faire qu'une seule fois par session Colab.

In [ ]:
# Téléchargement et installation d'OpenSTL (quelques minutes)
!git clone https://github.com/lsteffenel/OpenSTL
import os
os.chdir('/content/OpenSTL')
!python setup.py install

In [ ]:
# Installation des dépendances
!pip install -q lightning lightning-utilities iopath
!pip install -q fvcore PyWavelets imageio matplotlib scipy timm torch-optim einops scikit-image

**Piège GPU classique** : l'une des bibliothèques installées peut entraîner `pip` à **remplacer** la version de PyTorch préinstallée sur Colab (compilée pour GPU) par la version du dépôt général PyPI, compilée **pour CPU uniquement**. Symptôme : le message « Torch not compiled with CUDA enabled », même sur une machine GPU (T4).

La cellule suivante vérifie le support GPU après l'installation. Si tout est en ordre, continuez directement. Sinon, exécutez la cellule de réparation, puis redémarrez la session une seconde fois.

In [ ]:
# Vérification du support GPU
import torch
print("Version de torch :", torch.__version__)
print("Compilation CUDA :", torch.version.cuda)
print("GPU disponible   :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU détecté      :", torch.cuda.get_device_name(0))

In [ ]:
# Cellule de RÉPARATION — à n'exécuter QUE si « GPU disponible : False » ci-dessus.
# Elle réinstalle PyTorch dans sa version GPU, puis il faut redémarrer la session
# (Exécution -> Redémarrer la session) et re-exécuter la cellule os.chdir.
!pip install -q --force-reinstall torch torchvision --index-url https://download.pytorch.org/whl/cu126

# Note : l'index (cu126) correspond au CUDA des Colab récents ; si la
# réinstallation échoue, vérifiez la version CUDA de Colab avec :
# !nvcc --version

**Important** : après ces deux cellules, redémarrez la session Colab (menu *Exécution → Redémarrer la session*), puis re-exécutez uniquement la cellule ci-dessous (sans relancer les installations), avant de continuer.

In [ ]:
# Après le redémarrage de la session, re-exécutez cette cellule pour
# revenir dans le dossier d'OpenSTL — puis continuez à la suite.
import os
os.chdir('/content/OpenSTL')

## Étape 1 — Préparer vos données

Le scénario : vous disposez d'une collection de vidéos, rangées dans trois dossiers selon leur usage :

- `train/` : vidéos utilisées pour **apprendre** (l'essentiel des données) ;
- `val/` : vidéos utilisées pour **suivre** l'apprentissage pendant l'entraînement ;
- `test/` : vidéos gardées **à l'écart**, pour évaluer honnêtement le modèle final.

Dans cet exemple, les vidéos sont au format `.avi` (d'autres formats conviennent). Ce sont des extraits du jeu de données KTH (vidéos de personnes en mouvement), enregistrés en RVB.

L'idée de la préparation : découper chaque vidéo en petites séquences de 12 images consécutives — les 6 premières serviront d'**entrée** du modèle, les 6 suivantes de **cible** (ce que le modèle doit prédire).

### 1.1 Deux réglages à connaître

- `pre_seq_length` : nombre d'images **données** au modèle ;
- `aft_seq_length` : nombre d'images **à prédire**.

In [ ]:
# À ADAPTER : nombre d'images en entrée et en sortie
pre_seq_length = 6
aft_seq_length = 6

### 1.2 Extraction des séquences

La cellule ci-dessous contient deux fonctions utilitaires :

- `sample_frames` : lit une vidéo et la découpe en séquences de `pre_seq_length + aft_seq_length` images consécutives (toutes les 4 images de départ) ;
- `process_folder` : applique ce découpage à toutes les vidéos d'un dossier, puis remet les valeurs des pixels dans l'intervalle [0, 1] (indispensable pour un apprentissage stable).

Vous n'avez pas besoin de modifier ce code : exécutez-le simplement.

In [ ]:
import cv2
import numpy as np
import os

def sample_frames(video_path, num_frames=20):
    """Découpe une vidéo en séquences de num_frames images consécutives."""
    video = cv2.VideoCapture(video_path)
    total_frames = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
    print(video_path + " - " + str(total_frames) + " images")
    stacks = []
    for idx in range(0, total_frames - num_frames, 4):
        frames = []
        for i in range(num_frames):
            video.set(cv2.CAP_PROP_POS_FRAMES, idx + i)
            _, frame = video.read()
            frames.append(frame)
        # empile les images : T x C x H x L (temps, canaux RVB, hauteur, largeur)
        stack = np.stack(frames).transpose(0, 3, 1, 2)
        stacks.append(stack)
    video.release()
    return stacks

def process_folder(folder_path, pre_slen=10, aft_slen=10, suffix='.avi'):
    """Traite toutes les vidéos d'un dossier et renvoie (entrées, cibles)."""
    videos = []
    files = sorted(os.listdir(folder_path))
    for file in files:
        video_path = os.path.join(folder_path, file)
        if os.path.isfile(video_path) and file.endswith(suffix):
            videos.extend(sample_frames(video_path, pre_slen + aft_slen))
    data = np.array(videos)
    print("   -> tableau de forme", data.shape)

    # valeurs des pixels de [0, 255] vers [0, 1]
    if data.max() > 1.0:
        data = data.astype(np.float32) / 255.0

    # les pre_slen premières images = entrée, les suivantes = cible
    return data[:, :pre_slen], data[:, pre_slen:]

In [ ]:
# Traitement des trois dossiers (peut prendre un peu de temps)
dataset = {}
for folder in ['train', 'val', 'test']:
    print(f"Dossier {folder} :")
    data_x, data_y = process_folder('examples/' + folder,
                                    pre_slen=pre_seq_length,
                                    aft_slen=aft_seq_length,
                                    suffix='.avi')
    dataset['X_' + folder], dataset['Y_' + folder] = data_x, data_y

In [ ]:
train_x, train_y = dataset['X_train'], dataset['Y_train']

### 1.3 Vérification visuelle

Avant d'entraîner un modèle, on vérifie toujours que les données sont correctes. Affichons une séquence d'entrée puis la séquence cible correspondante :

In [ ]:

# Forme du tableau : B x T x C x H x L
#   B : nombre de séquences, T : nombre d'images, C : canaux RVB, H/L : hauteur, largeur
print("Entrées :", train_x.shape)
print("Cibles  :", train_y.shape)

In [ ]:
from openstl.utils import show_video_line

# Les images données au modèle (le "passé")
example_idx = 1
show_video_line(train_x[example_idx], ncols=pre_seq_length,
                vmax=0.6, cbar=False, out_path=None, format='png', use_rgb=True)

In [ ]:
# Les images à prédire (le "futur")
show_video_line(train_y[example_idx], ncols=aft_seq_length,
                vmax=0.6, cbar=False, out_path=None, format='png', use_rgb=True)

## Étape 2 — Organiser les données pour l'entraînement

PyTorch (le framework sous-jacent d'OpenSTL) lit les données via un objet appelé *Dataset*. La classe ci-dessous est volontairement minimale : elle enveloppe nos tableaux NumPy pour que PyTorch sache les parcourir. Elle remet aussi les valeurs en `float32`, le format de calcul attendu par le modèle.

Pour un vrai projet, deux remarques :

- Les vidéos RVB sont déjà dans [0, 1] après l'étape 1 : rien de plus à faire. Pour d'autres types de données (cartes de température, d'humidité...), une **normalisation** (retrait de la moyenne, division par l'écart-type) est fortement recommandée.
- Le `batch_size` (nombre de séquences traitées simultanément) vaut 1 ici pour rester léger en mémoire ; on l'augmente typiquement à 16 ou 32 sur un vrai projet.

In [ ]:
import torch
from torch.utils.data import Dataset

class CustomDataset(Dataset):
    """Enveloppe minimale : rend un tableau NumPy lisible par PyTorch."""
    def __init__(self, X, Y, data_name='custom', normalize=False):
        # OpenSTL lit ces attributs sur le dataset : ils doivent exister,
        # même si aucune normalisation n'est appliquée (mean = std = None).
        self.mean = None
        self.std = None
        self.data_name = data_name

        if normalize:
            # normalisation par canal : (x - moyenne) / écart-type
            # utile pour des données non-image (températures, humidité...)
            mean = X.mean(axis=(0, 1, 3, 4)).reshape(1, 1, -1, 1, 1)
            std  = X.std(axis=(0, 1, 3, 4)).reshape(1, 1, -1, 1, 1)
            X = (X - mean) / std
            Y = (Y - mean) / std
            self.mean, self.std = mean, std

        self.X = X          # séquences d'entrée  (B x T x C x H x L)
        self.Y = Y          # séquences cibles

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, index):
        data  = torch.tensor(self.X[index]).float()
        labels = torch.tensor(self.Y[index]).float()
        return data, labels

In [ ]:
# À ADAPTER : taille des lots (16 ou 32 pour un vrai projet)
batch_size = 1

train_set = CustomDataset(X=dataset['X_train'], Y=dataset['Y_train'])
val_set   = CustomDataset(X=dataset['X_val'],   Y=dataset['Y_val'])
test_set  = CustomDataset(X=dataset['X_test'],  Y=dataset['Y_test'])

dataloader_train = torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=True, pin_memory=True)
dataloader_val   = torch.utils.data.DataLoader(val_set,   batch_size=batch_size, shuffle=True, pin_memory=True)
dataloader_test  = torch.utils.data.DataLoader(test_set,  batch_size=batch_size, shuffle=True, pin_memory=True)

## Étape 3 — Configurer, entraîner, évaluer

### 3.1 Les deux blocs de réglages

Toute l'expérience se pilote avec deux dictionnaires. C'est **ici** que vous interviendrez le plus.

Le premier, `custom_training_config`, décrit l'entraînement :

| Clé | Rôle | Conseil |
|---|---|---|
| `epoch` | nombre de passages sur les données | 25 ici pour la démo ; comptez ~100 pour un vrai projet |
| `lr` | vitesse d'apprentissage | laisser 0.001 sauf problème |
| `metrics` | erreurs calculées | MSE et MAE, des erreurs moyennes en pixels |
| `early_stop_epoch` | arrêt anticipé si plus de progrès | évite de perdre du temps |
| `ex_name` | nom du dossier de résultats | changez-le pour chaque essai |

Le second, `custom_model_config`, décrit le modèle lui-même. Nous utilisons **SimVP** avec le module `gSTA` : c'est le modèle de référence d'OpenSTL, un bon point de départ. Ses paramètres internes (`N_S`, `N_T`, `hid_S`, `hid_T`) contrôlent la taille du réseau : dans un premier temps, gardez les valeurs par défaut ci-dessous.

In [ ]:
custom_training_config = {
    'pre_seq_length': pre_seq_length,       # images en entrée
    'aft_seq_length': aft_seq_length,       # images à prédire
    'total_length': pre_seq_length + aft_seq_length,
    'batch_size': batch_size,
    'val_batch_size': batch_size,
    'epoch': 25,                            # À ADAPTER : 100 pour un vrai projet
    'lr': 0.001,
    'metrics': ['mse', 'mae'],              # erreurs mesurées à chaque époque
    'early_stop_epoch': 10,                 # arrêt si plus de progrès pendant 10 époques
    'ex_name': 'custom_exp',                # À ADAPTER : nom du dossier de résultats
    'dataname': 'custom',
    'in_shape': list(dataset['X_train'].shape[1:]),  # T x C x H x L, déduit automatiquement
}

custom_model_config = {
    'method': 'SimVP',     # modèle de référence d'OpenSTL (bon point de départ)
    'model_type': 'gSTA', # variante interne du modèle
    # Paramètres de taille du réseau (garder les valeurs par défaut au début) :
    'N_S': 2,        # blocs spatiaux (analyse des images)
    'N_T': 6,        # blocs temporels (analyse de l'évolution)
    'hid_S': 32,     # neurones par bloc spatial
    'hid_T': 128,    # neurones par bloc temporel
}

### 3.2 Assemblage de l'expérience

La cellule ci-dessous construit l'objet `exp` qui centralise tout : données, modèle, optimiseur, suivi. Elle n'a pas besoin d'être modifiée.

In [ ]:
from openstl.api import BaseExperiment
from openstl.utils import create_parser, default_parser

# récupération des réglages par défaut de la bibliothèque
args = create_parser().parse_args([])
config = args.__dict__
default_values = default_parser()
for attribute in default_values.keys():
    if config[attribute] is None:
        config[attribute] = default_values[attribute]

# application de NOS réglages
config.update(custom_training_config)
config.update(custom_model_config)

exp = BaseExperiment(args, dataloaders=(dataloader_train, dataloader_val, dataloader_test),
                     strategy='auto')

### 3.3 Lancement

Il ne reste qu'à lancer l'entraînement, puis le test. Les résultats (prédictions sur le jeu de test) sont sauvegardés automatiquement dans `work_dirs/custom_exp/saved/`.

In [ ]:
print('>'*35 + ' entraînement ' + '<'*35)
exp.train()

print('>'*35 + ' test ' + '<'*35)
exp.test()

## Étape 4 — Regarder ce que le modèle a prédit

C'est l'étape la plus parlante : comparer, image par image, la **réalité** (ce qui s'est réellement passé après les 6 images d'entrée) et la **prédiction** du modèle. On peut aussi générer un GIF animé qui superpose les deux.

In [ ]:
import numpy as np
from openstl.utils import show_video_line

# Chargement des résultats sauvegardés par exp.test()
inputs = np.load('./work_dirs/custom_exp/saved/inputs.npy')  # images données
preds  = np.load('./work_dirs/custom_exp/saved/preds.npy')  # images prédites
trues  = np.load('./work_dirs/custom_exp/saved/trues.npy')  # images réelles

print("Réalité :", trues.shape, "| Prédiction :", preds.shape)

In [ ]:
example_idx = 10   # À ADAPTER : choisir une autre séquence (0 à nb-1)

# La réalité (images réellement observées)
show_video_line(trues[example_idx], ncols=aft_seq_length,
                vmax=0.6, cbar=False, out_path=None, format='png', use_rgb=True)

In [ ]:
# La prédiction du modèle — comparez image par image avec la cellule ci-dessus
show_video_line(preds[example_idx], ncols=aft_seq_length,
                vmax=0.6, cbar=False, out_path=None, format='png', use_rgb=True)

### 4.1 Générer un GIF animé

Le GIF superpose entrée, réalité et prédiction — pratique pour une présentation ou un rapport. Le fichier `example.gif` apparaît dans le panneau de fichiers de Colab (icône dossier, à gauche).

In [ ]:
from openstl.utils import show_video_gif_multiple

show_video_gif_multiple(inputs[example_idx], trues[example_idx], preds[example_idx],
                         use_rgb=True, out_path='example.gif')

## Récapitulatif

Ce que vous avez fait, en quatre étapes :

1. **Données** : des vidéos brutes → des tableaux de séquences (entrée : 6 images, cible : 6 images), valeurs normalisées dans [0, 1] ;
2. **Organisation** : les tableaux sont enveloppés dans un `CustomDataset` puis des `DataLoader` (train / val / test) ;
3. **Entraînement** : deux dictionnaires de réglages (entraînement + modèle) pilotent tout l'expérience via `BaseExperiment` ;
4. **Évaluation visuelle** : comparaison image par image entre prédiction et réalité, export GIF.

**Pour aller plus loin avec vos propres données** (imagerie satellite, sorties de modèle, séries de cartes météo), la démarche est identique : il suffit de produire des tableaux `X` (entrées) et `Y` (cibles) de forme `B x T x C x H x L`, puis d'ajuster `in_shape` et, si vos variables ne sont pas des images RVB, d'ajouter une normalisation dans le `CustomDataset`. Attention aussi aux tailles d'images : un format multiple de 16 évite les erreurs d'arrondi dans les réseaux convolutifs.